# Credit Risk Scoring - Arquitectura Medallion

## Objetivo
Desarrollar un modelo de scoring de riesgo crediticio utilizando el dataset de Kaggle [Credit Risk Dataset](https://www.kaggle.com/datasets/laotse/credit-risk-dataset).

## Arquitectura

### 🥉 Bronze Layer
- Ingesta de datos raw desde Kaggle
- Mínima transformación
- Registro de timestamp de ingesta

### 🥈 Silver Layer
- Limpieza de datos
- Validación de tipos de datos
- Manejo de valores nulos
- Eliminación de duplicados
- Estandarización de formatos

### 🥇 Gold Layer
- Feature engineering
- Encoding de variables categóricas
- Preparación para ML
- Train/test split

## Modelos a Comparar
1. Logistic Regression
2. Random Forest
3. Gradient Boosting (GBT)
4. XGBoost
5. LightGBM

**Métrica de selección:** AUC (Area Under ROC Curve)

## Entregables
- Modelo campeón con mejor AUC
- Score de riesgo crediticio por cliente
- Comparación de rendimiento de modelos

In [0]:
%pip install kaggle xgboost lightgbm optuna category-encoders imbalanced-learn --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *
import mlflow
import mlflow.sklearn
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report, f1_score, precision_score, recall_score
import category_encoders as ce
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas correctamente")

In [0]:
# El dataset está disponible públicamente en Kaggle
# Descargaremos el archivo y lo cargaremos
import os
import zipfile

# Crear directorio temporal si no existe
temp_dir = "/tmp/credit_risk/"
os.makedirs(temp_dir, exist_ok=True)

# El dataset se puede descargar usando kaggle API o manualmente
# Por simplicidad, usaremos un enfoque directo
try:
    # Intentar descargar con kaggle API
    !kaggle datasets download -d laotse/credit-risk-dataset -p /tmp/credit_risk/ --unzip
    print("✅ Dataset descargado con Kaggle API")
except:
    print("⚠️ Kaggle API no configurada. Por favor:")
    print("1. Descarga el dataset manualmente desde: https://www.kaggle.com/datasets/laotse/credit-risk-dataset")
    print("2. Sube el archivo credit_risk_dataset.csv a /tmp/credit_risk/")
    print("3. O configura tu Kaggle API token en ~/.kaggle/kaggle.json")

## 🥉 Bronze Layer

Cargamos los datos raw desde el CSV y los guardamos en una tabla Delta con mínima transformación.

In [0]:
# Leer el CSV
csv_path = "/tmp/credit_risk/credit_risk_dataset.csv"

# Cargar con pandas primero para inspección rápida
pdf = pd.read_csv(csv_path)
print(f"Dataset shape: {pdf.shape}")
print(f"\nColumnas: {pdf.columns.tolist()}")
print(f"\nPrimeras filas:")
display(pdf.head())

# Convertir a Spark DataFrame para procesamiento distribuido
df_bronze = spark.createDataFrame(pdf)

# Agregar timestamp de ingesta
df_bronze = df_bronze.withColumn("ingestion_timestamp", F.current_timestamp())

# Crear esquema si no existe
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

# Guardar como tabla Delta en Bronze layer
df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze.credit_risk_raw")

print(f"\n✅ Tabla bronze.credit_risk_raw creada con {df_bronze.count()} registros")

## 🥈 Silver Layer

Limpieza de datos, manejo de nulos, validación de tipos, y eliminación de duplicados.

In [0]:
# Leer tabla Bronze
df_silver = spark.table("bronze.credit_risk_raw")

print("=== ANÁLISIS DE CALIDAD DE DATOS ===")
print(f"\nTotal registros: {df_silver.count()}")
print(f"\nEsquema:")
df_silver.printSchema()

# Eliminar columna ingestion_timestamp antes de convertir a pandas
if 'ingestion_timestamp' in df_silver.columns:
    df_silver = df_silver.drop('ingestion_timestamp')

# Convertir a pandas para análisis detallado
pdf_analysis = df_silver.toPandas()

print("\n=== VALORES NULOS ===")
null_counts = pdf_analysis.isnull().sum()
print(null_counts[null_counts > 0])

print("\n=== ESTADÍSTICAS DESCRIPTIVAS ===")
display(pdf_analysis.describe())

print("\n=== DISTRIBUCIÓN DE LA VARIABLE TARGET ===")
if 'loan_status' in pdf_analysis.columns:
    print(pdf_analysis['loan_status'].value_counts())
elif 'cb_person_default_on_file' in pdf_analysis.columns:
    print(pdf_analysis['cb_person_default_on_file'].value_counts())

In [0]:
# Eliminar columna de timestamp de ingesta para procesamiento
df_silver = df_silver.drop("ingestion_timestamp")

# Identificar duplicados
initial_count = df_silver.count()
df_silver = df_silver.dropDuplicates()
final_count = df_silver.count()
print(f"Duplicados eliminados: {initial_count - final_count}")

# Manejo de valores nulos
# Para columnas numéricas: imputar con la mediana
numeric_cols = [f.name for f in df_silver.schema.fields if isinstance(f.dataType, (IntegerType, DoubleType, FloatType, LongType))]

for col in numeric_cols:
    # Calcular mediana
    median_val = df_silver.approxQuantile(col, [0.5], 0.01)[0]
    # Rellenar nulos con mediana
    df_silver = df_silver.fillna({col: median_val})

print(f"\nColumnas numéricas procesadas: {numeric_cols}")

# Para columnas categóricas: rellenar con 'Unknown' o la moda
string_cols = [f.name for f in df_silver.schema.fields if isinstance(f.dataType, StringType)]

for col in string_cols:
    df_silver = df_silver.fillna({col: 'Unknown'})

print(f"Columnas categóricas procesadas: {string_cols}")

# Crear esquema Silver
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# Guardar tabla Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.credit_risk_cleaned")

print(f"\n✅ Tabla silver.credit_risk_cleaned creada con {df_silver.count()} registros")

## 🥇 Gold Layer

Análisis Exploratorio de Datos (EDA) y preparación de features para Machine Learning.

In [0]:
# Cargar datos de Silver
df_gold = spark.table("silver.credit_risk_cleaned")
pdf_eda = df_gold.toPandas()

print("=== EXPLORATORY DATA ANALYSIS ===")
print(f"\nShape: {pdf_eda.shape}")
print(f"\nInfo:")
print(pdf_eda.info())

# Identificar variable target
target_col = None
for col in ['loan_status', 'cb_person_default_on_file', 'loan_intent']:
    if col in pdf_eda.columns:
        if pdf_eda[col].nunique() == 2:  # Binary classification
            target_col = col
            break

if target_col:
    print(f"\n✅ Variable target identificada: {target_col}")
    print(f"Distribución:\n{pdf_eda[target_col].value_counts()}")
else:
    print("\n⚠️ No se identificó variable target binaria automáticamente")

# Tipos de features
print("\n=== TIPOS DE FEATURES ===")
feature_types = []
for col in pdf_eda.columns:
    if col == target_col:
        continue
    dtype = pdf_eda[col].dtype
    if dtype in ['int64', 'float64']:
        feature_types.append([col, 'numeric'])
    else:
        feature_types.append([col, 'categorical'])

feature_df = pd.DataFrame(feature_types, columns=['Feature', 'Type'])
display(feature_df)

In [0]:
# Seleccionar solo columnas numéricas para correlación
numeric_cols = pdf_eda.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) > 1:
    # Correlograma
    plt.figure(figsize=(12, 10))
    correlation_matrix = pdf_eda[numeric_cols].corr()
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title('Matriz de Correlación de Features Numéricas')
    plt.tight_layout()
    plt.show()
    
    # Si hay target numérico, mostrar correlaciones con target
    if target_col and target_col in numeric_cols:
        print("\n=== CORRELACIÓN CON TARGET ===")
        target_corr = correlation_matrix[target_col].sort_values(ascending=False)
        print(target_corr)
        
        # Top features correlacionadas con target
        top_features = target_corr[target_corr.index != target_col].head(3).index.tolist()
        
        # Visualizaciones de top features vs target
        fig, axes = plt.subplots(1, min(3, len(top_features)), figsize=(15, 4))
        if len(top_features) == 1:
            axes = [axes]
        
        for idx, feature in enumerate(top_features[:3]):
            if len(top_features) > 1:
                ax = axes[idx]
            else:
                ax = axes[0]
            pdf_eda.plot.scatter(x=feature, y=target_col, alpha=0.5, ax=ax)
            ax.set_title(f'{feature} vs {target_col}')
        
        plt.tight_layout()
        plt.show()

# Distribución de clases
if target_col:
    plt.figure(figsize=(8, 6))
    pdf_eda[target_col].value_counts().plot(kind='bar', color=['skyblue', 'salmon'])
    plt.title(f'Distribución de {target_col}')
    plt.xlabel(target_col)
    plt.ylabel('Frecuencia')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [0]:
# Preparar datos para ML
pdf_gold = pdf_eda.copy()

# Identificar target
if target_col is None:
    # Intentar encontrar target manualmente
    print("⚠️ Especifica la columna target")
    print(f"Columnas disponibles: {pdf_gold.columns.tolist()}")
else:
    print(f"Target column: {target_col}")
    
    # Separar features y target
    X = pdf_gold.drop(columns=[target_col])
    y = pdf_gold[target_col]
    
    # Codificar target si es categórico
    if y.dtype == 'object':
        le = LabelEncoder()
        y = le.fit_transform(y)
        print(f"Target encoding: {dict(enumerate(le.classes_))}")
    
    # Identificar columnas categóricas y numéricas
    categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
    numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    
    print(f"\nCategorical features ({len(categorical_cols)}): {categorical_cols}")
    print(f"Numerical features ({len(numerical_cols)}): {numerical_cols}")
    
    # One-Hot Encoding para variables categóricas
    if categorical_cols:
        X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
        print(f"\nFeatures después de encoding: {X.shape[1]}")
    
    # Guardar dataset procesado
    pdf_gold_final = X.copy()
    pdf_gold_final[target_col] = y
    
    print(f"\n✅ Feature engineering completado")
    print(f"Final shape: {pdf_gold_final.shape}")

In [0]:
# Train/Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nDistribución en training:")
print(pd.Series(y_train).value_counts())
print(f"\nDistribución en test:")
print(pd.Series(y_test).value_counts())

# Convertir a Spark DataFrame y guardar
spark_gold = spark.createDataFrame(pdf_gold_final)

# Crear esquema Gold
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# Guardar tabla Gold
spark_gold.write.format("delta").mode("overwrite").saveAsTable("gold.credit_risk_features")

print(f"\n✅ Tabla gold.credit_risk_features creada con {spark_gold.count()} registros")

# Guardar train/test splits para referencia
train_df = X_train.copy()
train_df[target_col] = y_train
test_df = X_test.copy()
test_df[target_col] = y_test

print("\n✅ Train/Test splits preparados en memoria")

## 🤖 Model Training

Entrenamos múltiples modelos de clasificación y los comparamos por AUC.

In [0]:
# Configurar experimento MLflow
experiment_name = "/Users/" + spark.sql("SELECT current_user()").collect()[0][0] + "/credit_risk_scoring"
mlflow.set_experiment(experiment_name)

print(f"✅ MLflow experiment configurado: {experiment_name}")

# Preparar datos para sklearn
from sklearn.preprocessing import StandardScaler

# Escalar features numéricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features escaladas con StandardScaler")

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
import time

with mlflow.start_run(run_name="Logistic_Regression") as run:
    start_time = time.time()
    
    # Entrenar modelo
    lr_model = LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced'  # Manejar desbalance
    )
    lr_model.fit(X_train_scaled, y_train)
    
    # Predecir
    y_pred = lr_model.predict(X_test_scaled)
    y_pred_proba = lr_model.predict_proba(X_test_scaled)[:, 1]
    
    # Calcular métricas
    auc_score = roc_auc_score(y_test, y_pred_proba)
    
    # Logging en MLflow
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("auc", auc_score)
    mlflow.log_metric("training_time", time.time() - start_time)
    
    # Log modelo con signature
    from mlflow.models.signature import infer_signature
    signature = infer_signature(X_train_scaled, lr_model.predict_proba(X_train_scaled))
    mlflow.sklearn.log_model(lr_model, "model", signature=signature)
    
    print(f"Logistic Regression AUC: {auc_score:.4f}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

In [0]:
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="Random_Forest") as run:
    start_time = time.time()
    
    # Entrenar modelo
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=10,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )
    rf_model.fit(X_train_scaled, y_train)
    
    # Predecir
    y_pred = rf_model.predict(X_test_scaled)
    y_pred_proba = rf_model.predict_proba(X_test_scaled)[:, 1]
    
    # Calcular métricas
    auc_score = roc_auc_score(y_test, y_pred_proba)
    
    # Logging en MLflow
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 10)
    mlflow.log_metric("auc", auc_score)
    mlflow.log_metric("training_time", time.time() - start_time)
    
    # Log modelo
    signature = infer_signature(X_train_scaled, rf_model.predict_proba(X_train_scaled))
    mlflow.sklearn.log_model(rf_model, "model", signature=signature)
    
    print(f"Random Forest AUC: {auc_score:.4f}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

In [0]:
from sklearn.ensemble import GradientBoostingClassifier

with mlflow.start_run(run_name="Gradient_Boosting") as run:
    start_time = time.time()
    
    # Entrenar modelo
    gb_model = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    )
    gb_model.fit(X_train_scaled, y_train)
    
    # Predecir
    y_pred = gb_model.predict(X_test_scaled)
    y_pred_proba = gb_model.predict_proba(X_test_scaled)[:, 1]
    
    # Calcular métricas
    auc_score = roc_auc_score(y_test, y_pred_proba)
    
    # Logging en MLflow
    mlflow.log_param("model_type", "GradientBoosting")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 5)
    mlflow.log_metric("auc", auc_score)
    mlflow.log_metric("training_time", time.time() - start_time)
    
    # Log modelo
    signature = infer_signature(X_train_scaled, gb_model.predict_proba(X_train_scaled))
    mlflow.sklearn.log_model(gb_model, "model", signature=signature)
    
    print(f"Gradient Boosting AUC: {auc_score:.4f}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

In [0]:
from xgboost import XGBClassifier

with mlflow.start_run(run_name="XGBoost") as run:
    start_time = time.time()
    
    # Entrenar modelo
    xgb_model = XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    )
    xgb_model.fit(X_train_scaled, y_train)
    
    # Predecir
    y_pred = xgb_model.predict(X_test_scaled)
    y_pred_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]
    
    # Calcular métricas
    auc_score = roc_auc_score(y_test, y_pred_proba)
    
    # Logging en MLflow
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 5)
    mlflow.log_metric("auc", auc_score)
    mlflow.log_metric("training_time", time.time() - start_time)
    
    # Log modelo
    signature = infer_signature(X_train_scaled, xgb_model.predict_proba(X_train_scaled))
    mlflow.sklearn.log_model(xgb_model, "model", signature=signature)
    
    print(f"XGBoost AUC: {auc_score:.4f}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

In [0]:
from lightgbm import LGBMClassifier

with mlflow.start_run(run_name="LightGBM") as run:
    start_time = time.time()
    
    # Entrenar modelo
    lgbm_model = LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        verbose=-1
    )
    lgbm_model.fit(X_train_scaled, y_train)
    
    # Predecir
    y_pred = lgbm_model.predict(X_test_scaled)
    y_pred_proba = lgbm_model.predict_proba(X_test_scaled)[:, 1]
    
    # Calcular métricas
    auc_score = roc_auc_score(y_test, y_pred_proba)
    
    # Logging en MLflow
    mlflow.log_param("model_type", "LightGBM")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 5)
    mlflow.log_metric("auc", auc_score)
    mlflow.log_metric("training_time", time.time() - start_time)
    
    # Log modelo
    signature = infer_signature(X_train_scaled, lgbm_model.predict_proba(X_train_scaled))
    mlflow.sklearn.log_model(lgbm_model, "model", signature=signature)
    
    print(f"LightGBM AUC: {auc_score:.4f}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

print("\n✅ Todos los modelos entrenados y registrados en MLflow")

## 🏆 Model Comparison

Comparamos todos los modelos por AUC y seleccionamos el campeón.

In [0]:
# Buscar todos los runs del experimento
experiment = mlflow.get_experiment_by_name(experiment_name)
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Ordenar por AUC descendente
runs_sorted = runs.sort_values('metrics.auc', ascending=False)

# Mostrar comparación
comparison_df = runs_sorted[['tags.mlflow.runName', 'metrics.auc', 'metrics.training_time']].copy()
comparison_df.columns = ['Model', 'AUC', 'Training Time (s)']
comparison_df = comparison_df.reset_index(drop=True)

print("=== COMPARACIÓN DE MODELOS ===")
display(comparison_df)

# Identificar mejor modelo
best_run = runs_sorted.iloc[0]
best_model_name = best_run['tags.mlflow.runName']
best_auc = best_run['metrics.auc']
best_run_id = best_run['run_id']

print(f"\n🏆 MODELO CAMPEÓN: {best_model_name}")
print(f"AUC: {best_auc:.4f}")
print(f"Run ID: {best_run_id}")

In [0]:
# Recrear predicciones para cada modelo
models_dict = {
    'Logistic Regression': lr_model,
    'Random Forest': rf_model,
    'Gradient Boosting': gb_model,
    'XGBoost': xgb_model,
    'LightGBM': lgbm_model
}

plt.figure(figsize=(10, 8))

for name, model in models_dict.items():
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Comparación de Modelos', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Curvas ROC generadas para todos los modelos")

In [0]:
# Cargar el mejor modelo desde MLflow
best_model_uri = f"runs:/{best_run_id}/model"
champion_model = mlflow.sklearn.load_model(best_model_uri)

print(f"✅ Modelo campeón cargado: {best_model_name}")
print(f"AUC en test set: {best_auc:.4f}")

In [0]:
# Cargar todos los datos de Gold layer
df_all_clients = spark.table("gold.credit_risk_features").toPandas()

# Separar features y target
if target_col in df_all_clients.columns:
    X_all = df_all_clients.drop(columns=[target_col])
    y_all = df_all_clients[target_col]
else:
    X_all = df_all_clients
    y_all = None

# Escalar features
X_all_scaled = scaler.transform(X_all)

# Generar scores de riesgo (probabilidad de default/riesgo)
risk_scores = champion_model.predict_proba(X_all_scaled)[:, 1]
risk_predictions = champion_model.predict(X_all_scaled)

# Crear DataFrame de resultados
scoring_results = pd.DataFrame({
    'client_id': range(len(risk_scores)),
    'risk_score': risk_scores,
    'risk_prediction': risk_predictions,
    'risk_category': pd.cut(risk_scores, 
                            bins=[0, 0.3, 0.6, 1.0], 
                            labels=['Low Risk', 'Medium Risk', 'High Risk'])
})

# Añadir target real si existe
if y_all is not None:
    scoring_results['actual_default'] = y_all.values

print("=== CREDIT RISK SCORING RESULTS ===")
print(f"\nTotal clientes: {len(scoring_results)}")
print(f"\nDistribución de categorías de riesgo:")
print(scoring_results['risk_category'].value_counts())
print(f"\nEstadísticas de Risk Score:")
print(scoring_results['risk_score'].describe())

# Mostrar muestra
print(f"\nMuestra de resultados:")
display(scoring_results.head(20))

In [0]:
# Convertir a Spark DataFrame
scoring_spark = spark.createDataFrame(scoring_results)

# Guardar en Gold layer
scoring_spark.write.format("delta").mode("overwrite").saveAsTable("gold.credit_risk_scores")

print(f"✅ Tabla gold.credit_risk_scores creada con {len(scoring_results)} registros")

# Visualización de distribución de scores
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma de scores
axes[0].hist(scoring_results['risk_score'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Risk Score', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribución de Risk Scores', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Bar chart de categorías
scoring_results['risk_category'].value_counts().plot(kind='bar', ax=axes[1], color=['green', 'orange', 'red'])
axes[1].set_xlabel('Risk Category', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Distribución de Categorías de Riesgo', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("🎉 PROYECTO COMPLETADO")
print("="*60)
print(f"\n🏆 Modelo Campeón: {best_model_name}")
print(f"🎯 AUC Score: {best_auc:.4f}")
print(f"📊 Total Clientes Scored: {len(scoring_results)}")
print(f"\n💾 Tablas Creadas:")
print("   - bronze.credit_risk_raw")
print("   - silver.credit_risk_cleaned")
print("   - gold.credit_risk_features")
print("   - gold.credit_risk_scores")
print("="*60)